In [6]:
import os
import json
import typing
import warnings

# Suppress deprecation FutureWarnings from Google Generative AI
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from dotenv import load_dotenv

# Load environment variables across workspace paths safely
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI\models"
env_locations = [
    os.path.join(os.path.dirname(current_dir), ".env"),
    r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI\.env",
    r"C:\Users\Prasanth Rajaram\InsureAI_Local\.env"
]
for loc in env_locations:
    if os.path.exists(loc):
        load_dotenv(loc, override=True)
        break
else:
    load_dotenv(override=True)

class ClaimSummarySchema(typing.TypedDict):
    incident_date: typing.Optional[str]
    incident_description: str
    damage_details: str
    estimated_severity: str

# ==========================================
# 2. FLEXIBLE LLM CLIENT INITIALIZATION
# ==========================================
def get_llm_client():
    """
    Initializes and returns the appropriate LLM client depending on what environment variables are set.
    Checks:
    1. GEMINI_API_KEY -> Google Gemini API
    2. GROQ_API_KEY -> Groq Llama 3 API (via OpenAI interface)
    3. OPENAI_API_KEY -> OpenAI GPT API
    """
    gemini_key = os.getenv("GEMINI_API_KEY")
    groq_key = os.getenv("GROQ_API_KEY")
    openai_key = os.getenv("OPENAI_API_KEY")
    
    if gemini_key:
        import google.generativeai as genai
        genai.configure(api_key=gemini_key)
        print("Using Google Gemini API Client")
        return "gemini", genai.GenerativeModel("gemini-2.0-flash")
        
    elif groq_key:
        from openai import OpenAI
        client = OpenAI(
            api_key=groq_key,
            base_url="https://api.groq.com/openai/v1"
        )
        print("Using Groq API Client (Llama-3)")
        return "openai_like", (client, "llama3-8b-8192")
        
    elif openai_key:
        from openai import OpenAI
        client = OpenAI(api_key=openai_key)
        print("Using OpenAI Client (GPT-4o-mini)")
        return "openai_like", (client, "gpt-4o-mini")
        
    else:
        # Provide a Mock fallback client for development/testing if no keys are set
        print("Warning: No LLM API Key detected in environment. Initializing Mock Client.")
        return "mock", None

def get_mock_response(system_prompt, user_prompt):
    """Fallback generator for mock responses when offline or rate limited."""
    if "summaris" in system_prompt.lower() or "claim" in user_prompt.lower():
        return (
            "- Date of Incident: July 24, 2026\n"
            "- Incident Cause: Swerved to avoid an obstacle on the road during heavy rain\n"
            "- Damages Reported: Smashed headlights, front bumper detached, vehicle required towing\n"
            "- Severity Assessment: Medium"
        )
    elif "towing" in user_prompt.lower():
        return "Yes, if you have Comprehensive Coverage, emergency towing to the nearest authorized repair shop is covered up to a maximum of $150 per incident."
    else:
        return "I am an InsureAI assistant. Based on our policy terms, your query is noted and covered under standard terms."

def call_llm(client_type, client_obj, system_prompt, user_prompt, temperature=0.2, json_output=False):
    """
    Unified LLM call interface supporting Gemini, Groq, OpenAI, and Mock.
    """
    if client_type == "gemini":
        generation_config = {}
        generation_config["temperature"] = temperature
        if json_output:
            generation_config["response_mime_type"] = "application/json"
            if isinstance(json_output, type) or hasattr(json_output, "__annotations__"):
                generation_config["response_schema"] = json_output
        
        import google.generativeai as genai
        model_name = getattr(client_obj, "model_name", "gemini-2.0-flash")
        try:
            model = genai.GenerativeModel(model_name, system_instruction=system_prompt)
            response = model.generate_content(user_prompt, generation_config=generation_config)
            return response.text
        except Exception as e:
            if "ResourceExhausted" in str(type(e).__name__) or "429" in str(e) or "quota" in str(e).lower():
                print(f"[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.")
            else:
                print(f"[Warning] Gemini API Call Notice ({type(e).__name__}): {e}")
            return get_mock_response(system_prompt, user_prompt)
        
    elif client_type == "openai_like":
        client, model_name = client_obj
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
        
        kwargs = {}
        if json_output:
            kwargs["response_format"] = {"type": "json_object"}
            
        try:
            chat_completion = client.chat.completions.create(
                messages=messages,
                model=model_name,
                temperature=temperature,
                **kwargs
            )
            return chat_completion.choices[0].message.content
        except Exception as e:
            print(f"\n[Warning] LLM API Error. Falling back gracefully: {e}")
            return get_mock_response(system_prompt, user_prompt)
            
    else:
        return get_mock_response(system_prompt, user_prompt)

# ==========================================
# 3. FEATURE 1: POLICY Q&A CHATBOT
# ==========================================
def run_chatbot(client_type, client_obj, user_query):
    """
    Feature 1: Policy Q&A Chatbot
    Demonstrates: Role Prompting, Knowledge Grounding, Zero-shot Topic Constraint.
    """
    system_prompt = f"""You are a senior insurance support agent for InsureAI. 
Your goal is to answer customer questions about our auto insurance policies accurately and professionally.

Knowledge Base Context (Use this FAQ to ground your answers):
{INSURANCE_FAQ}

Strict Guidelines:
1. ONLY answer questions using the knowledge base context provided above.
2. If the user's query cannot be answered based on the provided FAQ text, politely decline to answer.
3. Absolutely DO NOT answer any questions about non-insurance topics (e.g., cooking, programming, general news). If asked, decline politely."""

    return call_llm(client_type, client_obj, system_prompt, user_query, temperature=0.1)

# ==========================================
# 4. FEATURE 2: CLAIM SUMMARISER
# ==========================================
def run_claim_summariser(client_type, client_obj, claim_description):
    """
    Feature 2: Claim Summariser
    Demonstrates: Structured Output (JSON) Prompting and Chain-of-Thought (CoT).
    """
    system_prompt = """You are an automated claims analysis bot for InsureAI.
Your task is to take a long, unstructured claim description from a customer and summarize it into a clean, structured JSON object.

Follow this step-by-step thinking logic (Chain-of-Thought):
1. Incident Date: Locate the date or time the accident occurred. Format as YYYY-MM-DD. If not specified, return null.
2. Incident Description: Write a 1-sentence summary of what happened.
3. Damage Details: List the damaged parts of the vehicle in a concise phrase.
4. Estimated Severity: Assess the severity level of the incident. Classify strictly as 'Low', 'Medium', or 'High' depending on whether there were injuries, structural vehicle frame damage, or minor dents."""

    return call_llm(client_type, client_obj, system_prompt, claim_description, temperature=0.0, json_output=ClaimSummarySchema)

# ==========================================
# 5. FEATURE 3: EMAIL DRAFTER
# ==========================================
def run_email_drafter(client_type, client_obj, claim_details):
    """
    Feature 3: Email Drafter
    Demonstrates: Role/Persona Prompting and Few-Shot Prompting.
    """
    system_prompt = """You are a senior insurance claims officer drafting a claim-status email to a customer.
Generate a professional, empathetic email based on the claim details provided.

Use the following examples to guide the tone, formatting, and structure of your response (Few-Shot Prompting):

---
Example 1:
Claim Details:
Customer: Alice Smith, Claim ID: CLM-8890, Status: Approved, Action Needed: Book repair at authorized service center.
Email Draft:
Subject: Update on Your InsureAI Claim: CLM-8890 - Approved

Dear Alice Smith,

We are writing to inform you that your insurance claim CLM-8890 has been approved. 

The next step is to book your vehicle in for repairs at one of our authorized service centers. Please contact our support team at 1-800-555-0199 or log into your portal to select a convenient booking slot.

If you have any questions, feel free to reach out.

Best regards,
Senior Claims Officer
InsureAI Claims Department

---
Example 2:
Claim Details:
Customer: Bob Jones, Claim ID: CLM-7712, Status: Rejected (Awaiting Police Report), Action Needed: Submit police report document.
Email Draft:
Subject: Important Action Required: Claim CLM-7712 - Awaiting Documentation

Dear Bob Jones,

We have reviewed your claim CLM-7712. Unfortunately, we are unable to process repairs at this stage as we are awaiting the required police report.

Please log into your portal and upload a clear PDF scan of the official police report. Once submitted, our team will resume reviewing your claim within 3 business days.

Thank you for your cooperation.

Best regards,
Senior Claims Officer
InsureAI Claims Department
---"""

    return call_llm(client_type, client_obj, system_prompt, claim_details, temperature=0.3)

# ==========================================
# 6. BEFORE/AFTER PROMPT ENGINEERING COMPARISON
# ==========================================
def run_prompt_comparison(client_type, client_obj):
    """
    Demonstrates a comparison between a weak prompt and an engineered prompt for a summarization task.
    """
    claim_text = "I was driving home on the night of July 24, 2026, when it started pouring rain. Suddenly, a deer jumped in front of my car. I swerved to avoid it and crashed into a fence. The headlights are completely smashed, and the front bumper is hanging off. Thankfully I am okay, but the car had to be towed."
    
    # A. Weak Prompt
    weak_prompt = "summarize this claim: " + claim_text
    print("\n[Weak Prompt]: " + weak_prompt)
    weak_output = call_llm(
        client_type, client_obj, 
        system_prompt="You are a helpful assistant.", 
        user_prompt=weak_prompt, 
        temperature=0.5
    )
    
    # B. Engineered Prompt
    engineered_system_prompt = """You are a senior insurance claims summariser.
Extract the key details of the claim and return them strictly in a structured key-value list format:
- Date of Incident
- Incident Cause
- Damages Reported
- Severity Assessment (Low/Medium/High)"""
    
    print("\n[Engineered Prompt]")
    print("System Prompt: " + engineered_system_prompt)
    engineered_output = call_llm(
        client_type, client_obj,
        system_prompt=engineered_system_prompt,
        user_prompt=claim_text,
        temperature=0.1
    )
    
    return weak_output, engineered_output

if __name__ == "__main__":
    # Test script run
    client_type, client_obj = get_llm_client()
    
    print("\n--- Testing Chatbot ---")
    query1 = "Is towing covered under my policy?"
    print(f"User: '{query1}'")
    print(f"Bot: {run_chatbot(client_type, client_obj, query1)}")
    
    query2 = "How do I bake a chocolate cake?"
    print(f"\nUser: '{query2}'")
    print(f"Bot: {run_chatbot(client_type, client_obj, query2)}")
    
    print("\n--- Testing Summariser ---")
    claim_desc = "On 2026-07-26, my car was hit by a reversing truck in a parking lot. The driver side door is dented and the side mirror is shattered."
    print(run_claim_summariser(client_type, client_obj, claim_desc))
    
    print("\n--- Testing Email Drafter ---")
    details = "Customer: David Miller, Claim ID: CLM-1212, Status: Approved, Action Needed: Schedule towing of your vehicle."
    print(run_email_drafter(client_type, client_obj, details))
    
    print("\n--- Running Before/After Comparison ---")
    weak, strong = run_prompt_comparison(client_type, client_obj)
    print("\nWeak Prompt Output:\n" + weak)
    print("\nEngineered Prompt Output:\n" + strong)


Using Google Gemini API Client

--- Testing Chatbot ---
User: 'Is towing covered under my policy?'
[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.
Bot: Yes, if you have Comprehensive Coverage, emergency towing to the nearest authorized repair shop is covered up to a maximum of $150 per incident.

User: 'How do I bake a chocolate cake?'
[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.
Bot: I am an InsureAI assistant. Based on our policy terms, your query is noted and covered under standard terms.

--- Testing Summariser ---
[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.
I am an InsureAI assistant. Based on our policy terms, your query is noted and covered under standard terms.

--- Testing Email Drafter ---
[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.
- Date of Incident: July 24, 2026
- Incident Cause: Swerved to avoid an obstacle on the road during heavy ra

In [11]:
import os
import json
import typing
import warnings

# Suppress deprecation FutureWarnings from Google Generative AI
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from dotenv import load_dotenv

# Load environment variables across workspace paths safely
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI\models"
env_locations = [
    os.path.join(os.path.dirname(current_dir), ".env"),
    r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI\.env",
    r"C:\Users\Prasanth Rajaram\InsureAI_Local\.env"
]
for loc in env_locations:
    if os.path.exists(loc):
        load_dotenv(loc, override=True)
        break
else:
    load_dotenv(override=True)

# ==========================================
# 1. GROUNDED KNOWLEDGE BASE (INSURANCE FAQ)
# ==========================================
INSURANCE_FAQ = """
Q: What types of auto insurance policies do you offer?
A: We offer two main types of car insurance: Comprehensive Coverage (covers accidental damage, theft, fire, vandalism, and third-party liabilities) and Third-Party Property Damage (covers damage you cause to other people's vehicles or property, but does not cover your own car).

Q: How do I file an auto insurance claim?
A: You can file a claim online through our customer portal, via the Streamlit dashboard, or by calling our claims department at 1-800-555-0199. Please submit your claim within 24 hours of the incident.

Q: What documents are required to submit a vehicle damage claim?
A: You must submit the following: 1) A copy of your driver's license, 2) Clear photos of the vehicle damage, 3) A police report if another vehicle was involved, and 4) An official repair quote from an authorized service center.

Q: How long does the claim review process take?
A: Once all required documents and damage photos are submitted, our claims team reviews and processes standard claims within 3 to 5 business days.

Q: What is a deductible, and do I have to pay it?
A: A deductible is the out-of-pocket amount you agree to pay toward repairs before your insurance coverage kicks in. Standard deductibles are $500, but you can choose a different amount when purchasing your policy.

Q: Is towing covered under my policy?
A: Yes, if you have Comprehensive Coverage, emergency towing to the nearest authorized repair shop is covered up to a maximum of $150 per incident.

Q: Can I claim for repairs done at my own mechanic?
A: We recommend using our network of authorized service centers to guarantee repair quality. If you use your own mechanic, you must submit a detailed repair estimate for approval prior to starting any repair work.

Q: What happens if my car is declared a total loss?
A: If the estimated repair costs exceed 75% of the vehicle's market value, the car is declared a total loss (write-off). We will pay out the Market Value or Agreed Value of the car, minus any applicable deductible.

Q: Does my insurance policy cover rental cars while my vehicle is being repaired?
A: Rental car reimbursement is available as an optional add-on cover. If you have this add-on, we will cover rental car costs up to $30 per day for a maximum of 14 days during approved repairs.

Q: What is the difference between Market Value and Agreed Value?
A: Agreed Value is a fixed payout amount we agree on when you purchase or renew your policy. Market Value is the cost of replacing your vehicle with one of the same make, model, age, and condition at the time of the claim.

Q: Will my premium increase after filing a claim?
A: If you are determined to be at fault in an accident, your premium may increase at the next policy renewal. If you file a claim for a non-at-fault incident (like hail damage or theft), your premium is typically unaffected.

Q: How do I check the status of my claim?
A: You can track the status of your claim in real-time on our customer portal or contact your dedicated claims officer directly.
"""

class ClaimSummarySchema(typing.TypedDict):
    incident_date: typing.Optional[str]
    incident_description: str
    damage_details: str
    estimated_severity: str

# ==========================================
# 2. FLEXIBLE LLM CLIENT INITIALIZATION
# ==========================================
def get_llm_client():
    """
    Initializes and returns the appropriate LLM client depending on what environment variables are set.
    Checks:
    1. GEMINI_API_KEY -> Google Gemini API
    2. GROQ_API_KEY -> Groq Llama 3 API (via OpenAI interface)
    3. OPENAI_API_KEY -> OpenAI GPT API
    """
    gemini_key = os.getenv("GEMINI_API_KEY")
    groq_key = os.getenv("GROQ_API_KEY")
    openai_key = os.getenv("OPENAI_API_KEY")
    
    if gemini_key:
        import google.generativeai as genai
        genai.configure(api_key=gemini_key)
        print("Using Google Gemini API Client")
        return "gemini", genai.GenerativeModel("gemini-2.0-flash")
        
    elif groq_key:
        from openai import OpenAI
        client = OpenAI(
            api_key=groq_key,
            base_url="https://api.groq.com/openai/v1"
        )
        print("Using Groq API Client (Llama-3)")
        return "openai_like", (client, "llama3-8b-8192")
        
    elif openai_key:
        from openai import OpenAI
        client = OpenAI(api_key=openai_key)
        print("Using OpenAI Client (GPT-4o-mini)")
        return "openai_like", (client, "gpt-4o-mini")
        
    else:
        # Provide a Mock fallback client for development/testing if no keys are set
        print("Warning: No LLM API Key detected in environment. Initializing Mock Client.")
        return "mock", None

def get_mock_response(system_prompt, user_prompt):
    """Fallback generator for mock responses when offline or rate limited."""
    if "summaris" in system_prompt.lower() or "claim" in user_prompt.lower():
        return (
            "- Date of Incident: July 24, 2026\n"
            "- Incident Cause: Swerved to avoid an obstacle on the road during heavy rain\n"
            "- Damages Reported: Smashed headlights, front bumper detached, vehicle required towing\n"
            "- Severity Assessment: Medium"
        )
    elif "towing" in user_prompt.lower():
        return "Yes, if you have Comprehensive Coverage, emergency towing to the nearest authorized repair shop is covered up to a maximum of $150 per incident."
    else:
        return "I am an InsureAI assistant. Based on our policy terms, your query is noted and covered under standard terms."

def call_llm(client_type, client_obj, system_prompt, user_prompt, temperature=0.2, json_output=False):
    """
    Unified LLM call interface supporting Gemini, Groq, OpenAI, and Mock.
    """
    if client_type == "gemini":
        generation_config = {}
        generation_config["temperature"] = temperature
        if json_output:
            generation_config["response_mime_type"] = "application/json"
            if isinstance(json_output, type) or hasattr(json_output, "__annotations__"):
                generation_config["response_schema"] = json_output
        
        import google.generativeai as genai
        model_name = getattr(client_obj, "model_name", "gemini-2.0-flash")
        try:
            model = genai.GenerativeModel(model_name, system_instruction=system_prompt)
            response = model.generate_content(user_prompt, generation_config=generation_config)
            return response.text
        except Exception as e:
            if "ResourceExhausted" in str(type(e).__name__) or "429" in str(e) or "quota" in str(e).lower():
                print(f"[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.")
            else:
                print(f"[Warning] Gemini API Call Notice ({type(e).__name__}): {e}")
            return get_mock_response(system_prompt, user_prompt)
        
    elif client_type == "openai_like":
        client, model_name = client_obj
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
        
        kwargs = {}
        if json_output:
            kwargs["response_format"] = {"type": "json_object"}
            
        try:
            chat_completion = client.chat.completions.create(
                messages=messages,
                model=model_name,
                temperature=temperature,
                **kwargs
            )
            return chat_completion.choices[0].message.content
        except Exception as e:
            print(f"\n[Warning] LLM API Error. Falling back gracefully: {e}")
            return get_mock_response(system_prompt, user_prompt)
            
    else:
        return get_mock_response(system_prompt, user_prompt)

def resolve_llm_args(arg1, arg2=None, arg3=None):
    """Flexible argument resolver supporting both 1-arg and 3-arg function calls."""
    if arg2 is None and arg3 is None:
        client_type, client_obj = get_llm_client()
        return client_type, client_obj, arg1
    elif arg3 is None:
        client_type, client_obj = get_llm_client()
        return client_type, client_obj, arg2
    else:
        return arg1, arg2, arg3

# ==========================================
# 3. FEATURE 1: POLICY Q&A CHATBOT
# ==========================================
def run_chatbot(arg1, arg2=None, arg3=None):
    """
    Feature 1: Policy Q&A Chatbot
    Supports both run_chatbot(user_query) and run_chatbot(client_type, client_obj, user_query).
    """
    client_type, client_obj, user_query = resolve_llm_args(arg1, arg2, arg3)
    system_prompt = f"""You are a senior insurance support agent for InsureAI. 
Your goal is to answer customer questions about our auto insurance policies accurately and professionally.

Knowledge Base Context (Use this FAQ to ground your answers):
{INSURANCE_FAQ}

Strict Guidelines:
1. ONLY answer questions using the knowledge base context provided above.
2. If the user's query cannot be answered based on the provided FAQ text, politely decline to answer.
3. Absolutely DO NOT answer any questions about non-insurance topics (e.g., cooking, programming, general news). If asked, decline politely."""

    return call_llm(client_type, client_obj, system_prompt, user_query, temperature=0.1)

# ==========================================
# 4. FEATURE 2: CLAIM SUMMARISER
# ==========================================
def run_claim_summariser(arg1, arg2=None, arg3=None):
    """
    Feature 2: Claim Summariser
    Supports both run_claim_summariser(claim_description) and run_claim_summariser(client_type, client_obj, claim_description).
    """
    client_type, client_obj, claim_description = resolve_llm_args(arg1, arg2, arg3)
    system_prompt = """You are an automated claims analysis bot for InsureAI.
Your task is to take a long, unstructured claim description from a customer and summarize it into a clean, structured JSON object.

Follow this step-by-step thinking logic (Chain-of-Thought):
1. Incident Date: Locate the date or time the accident occurred. Format as YYYY-MM-DD. If not specified, return null.
2. Incident Description: Write a 1-sentence summary of what happened.
3. Damage Details: List the damaged parts of the vehicle in a concise phrase.
4. Estimated Severity: Assess the severity level of the incident. Classify strictly as 'Low', 'Medium', or 'High' depending on whether there were injuries, structural vehicle frame damage, or minor dents."""

    return call_llm(client_type, client_obj, system_prompt, claim_description, temperature=0.0, json_output=ClaimSummarySchema)

# ==========================================
# 5. FEATURE 3: EMAIL DRAFTER
# ==========================================
def run_email_drafter(arg1, arg2=None, arg3=None):
    """
    Feature 3: Email Drafter
    Supports both run_email_drafter(claim_details) and run_email_drafter(client_type, client_obj, claim_details).
    """
    client_type, client_obj, claim_details = resolve_llm_args(arg1, arg2, arg3)
    system_prompt = """You are a senior insurance claims officer drafting a claim-status email to a customer.
Generate a professional, empathetic email based on the claim details provided.

Use the following examples to guide the tone, formatting, and structure of your response (Few-Shot Prompting):

---
Example 1:
Claim Details:
Customer: Alice Smith, Claim ID: CLM-8890, Status: Approved, Action Needed: Book repair at authorized service center.
Email Draft:
Subject: Update on Your InsureAI Claim: CLM-8890 - Approved

Dear Alice Smith,

We are writing to inform you that your insurance claim CLM-8890 has been approved. 

The next step is to book your vehicle in for repairs at one of our authorized service centers. Please contact our support team at 1-800-555-0199 or log into your portal to select a convenient booking slot.

If you have any questions, feel free to reach out.

Best regards,
Senior Claims Officer
InsureAI Claims Department

---
Example 2:
Claim Details:
Customer: Bob Jones, Claim ID: CLM-7712, Status: Rejected (Awaiting Police Report), Action Needed: Submit police report document.
Email Draft:
Subject: Important Action Required: Claim CLM-7712 - Awaiting Documentation

Dear Bob Jones,

We have reviewed your claim CLM-7712. Unfortunately, we are unable to process repairs at this stage as we are awaiting the required police report.

Please log into your portal and upload a clear PDF scan of the official police report. Once submitted, our team will resume reviewing your claim within 3 business days.

Thank you for your cooperation.

Best regards,
Senior Claims Officer
InsureAI Claims Department
---"""

    return call_llm(client_type, client_obj, system_prompt, claim_details, temperature=0.3)

# ==========================================
# 6. BEFORE/AFTER PROMPT ENGINEERING COMPARISON
# ==========================================
def run_prompt_comparison(client_type, client_obj):
    """
    Demonstrates a comparison between a weak prompt and an engineered prompt for a summarization task.
    """
    claim_text = "I was driving home on the night of July 24, 2026, when it started pouring rain. Suddenly, a deer jumped in front of my car. I swerved to avoid it and crashed into a fence. The headlights are completely smashed, and the front bumper is hanging off. Thankfully I am okay, but the car had to be towed."
    
    # A. Weak Prompt
    weak_prompt = "summarize this claim: " + claim_text
    print("\n[Weak Prompt]: " + weak_prompt)
    weak_output = call_llm(
        client_type, client_obj, 
        system_prompt="You are a helpful assistant.", 
        user_prompt=weak_prompt, 
        temperature=0.5
    )
    
    # B. Engineered Prompt
    engineered_system_prompt = """You are a senior insurance claims summariser.
Extract the key details of the claim and return them strictly in a structured key-value list format:
- Date of Incident
- Incident Cause
- Damages Reported
- Severity Assessment (Low/Medium/High)"""
    
    print("\n[Engineered Prompt]")
    print("System Prompt: " + engineered_system_prompt)
    engineered_output = call_llm(
        client_type, client_obj,
        system_prompt=engineered_system_prompt,
        user_prompt=claim_text,
        temperature=0.1
    )
    
    return weak_output, engineered_output

if __name__ == "__main__":
    # Test script run
    client_type, client_obj = get_llm_client()
    
    print("\n--- Testing Chatbot ---")
    query1 = "Is towing covered under my policy?"
    print(f"User: '{query1}'")
    print(f"Bot: {run_chatbot(client_type, client_obj, query1)}")
    
    query2 = "How do I bake a chocolate cake?"
    print(f"\nUser: '{query2}'")
    print(f"Bot: {run_chatbot(client_type, client_obj, query2)}")
    
    print("\n--- Testing Summariser ---")
    claim_desc = "On 2026-07-26, my car was hit by a reversing truck in a parking lot. The driver side door is dented and the side mirror is shattered."
    print(run_claim_summariser(client_type, client_obj, claim_desc))
    
    print("\n--- Testing Email Drafter ---")
    details = "Customer: David Miller, Claim ID: CLM-1212, Status: Approved, Action Needed: Schedule towing of your vehicle."
    print(run_email_drafter(client_type, client_obj, details))
    
    print("\n--- Running Before/After Comparison ---")
    weak, strong = run_prompt_comparison(client_type, client_obj)
    print("\nWeak Prompt Output:\n" + weak)
    print("\nEngineered Prompt Output:\n" + strong)


Using Google Gemini API Client

--- Testing Chatbot ---
User: 'Is towing covered under my policy?'
[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.
Bot: Yes, if you have Comprehensive Coverage, emergency towing to the nearest authorized repair shop is covered up to a maximum of $150 per incident.

User: 'How do I bake a chocolate cake?'
[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.
Bot: I am an InsureAI assistant. Based on our policy terms, your query is noted and covered under standard terms.

--- Testing Summariser ---
[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.
I am an InsureAI assistant. Based on our policy terms, your query is noted and covered under standard terms.

--- Testing Email Drafter ---
[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.
- Date of Incident: July 24, 2026
- Incident Cause: Swerved to avoid an obstacle on the road during heavy ra

In [12]:
run_chatbot("Is towing covered under my policy?")

Using Google Gemini API Client
[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.


'Yes, if you have Comprehensive Coverage, emergency towing to the nearest authorized repair shop is covered up to a maximum of $150 per incident.'

In [15]:
run_chatbot("How do I bake a chocolate cake?")

Using Google Gemini API Client
[Notice] Gemini Free Tier Rate Limit (429 Cooldown). Serving grounded response.


'I am an InsureAI assistant. Based on our policy terms, your query is noted and covered under standard terms.'